# CEG-WM Stage-A content-adaptive dual-branch V2 clean execution

Content V2 clean mechanism only, with a fixed 8 units/16 records: no attacks, complementarity, superiority, geometry, fixed-FPR calibration, Stage-A/content-chain completion, scientific self-promotion, or paper promotion; local/Colab results are engineering evidence only, and any returned ZIP+SHA requires external supervisor validation.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os

drive.mount('/content/drive')

def _required_secret(name):
    value = os.environ.pop(name, None)
    if value is None:
        value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f'missing required Colab Secret: {name}')
    return value

root_key = _required_secret('CEG_WM_ROOT_KEY')
hf_token = _required_secret('HF_TOKEN')
artifact_sink = Path('/content/drive/MyDrive/CEG-WM/stage_a_content_adaptive_dual_branch_v2_clean')
artifact_sink.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, re, subprocess, sys

repo = Path('/content/cegwm-stage-a-content-adaptive-dual-branch-v2-source')
if repo.exists():
    raise RuntimeError('detached checkout path already exists')
subprocess.run(['git', 'init', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/RICHAAARC/CEG-WM.git'], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', 'refs/heads/stage-a-content-adaptive-dual-branch-v2'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
resolved_exact = subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if re.fullmatch(r'[0-9a-f]{40}', resolved_exact) is None:
    raise RuntimeError('resolved content V2 branch head is not an exact revision')
if subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout:
    raise RuntimeError('execution checkout is not clean')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)


In [ ]:
local_work_root = Path('/content/cegwm-stage-a-content-adaptive-dual-branch-v2-local')
runner_env = dict(os.environ)
runner_env['CEG_WM_ROOT_KEY'] = root_key
runner_env['HF_TOKEN'] = hf_token
command = [sys.executable, '-m', 'experiments.run_content_adaptive_dual_branch_v2_clean', '--repo-root', str(repo), '--expected-exact', resolved_exact, '--local-work-root', str(local_work_root), '--artifact-sink', str(artifact_sink)]
run_id = None
runner_summary = None
try:
    process = subprocess.Popen(command, cwd=str(repo), env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    for line in process.stdout:
        if line.startswith('CEGWM_PROGRESS '):
            progress = json.loads(line.removeprefix('CEGWM_PROGRESS '))
            if set(progress) != {'run_id', 'committed', 'fixed_total', 'phase'}:
                raise RuntimeError('runner progress has an unexpected schema')
            candidate_run_id = progress['run_id']
            if re.fullmatch(r'content-adaptive-v2-[0-9a-f]{12}-[0-9a-f]{12}', candidate_run_id) is None:
                raise RuntimeError('runner progress has an invalid run identity')
            if not isinstance(progress['committed'], int) or isinstance(progress['committed'], bool) or not 0 <= progress['committed'] <= 8:
                raise RuntimeError('runner progress committed count is out of bounds')
            if progress['fixed_total'] != 8 or progress['phase'] not in {'identity_ready', 'resume_ready', 'unit_committed', 'checkpoint_published'}:
                raise RuntimeError('runner progress bounds differ')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            run_id = candidate_run_id
        elif line.startswith('CEGWM_SUMMARY '):
            runner_summary = json.loads(line.removeprefix('CEGWM_SUMMARY '))
            if set(runner_summary) != {'run_id', 'committed', 'fixed_total', 'rc', 'phase'}:
                raise RuntimeError('runner summary has an unexpected schema')
            candidate_run_id = runner_summary['run_id']
            if re.fullmatch(r'content-adaptive-v2-[0-9a-f]{12}-[0-9a-f]{12}', candidate_run_id) is None:
                raise RuntimeError('runner summary has an invalid run identity')
            if not isinstance(runner_summary['committed'], int) or isinstance(runner_summary['committed'], bool) or not 0 <= runner_summary['committed'] <= 8:
                raise RuntimeError('runner summary committed count is out of bounds')
            if runner_summary['fixed_total'] != 8 or runner_summary['rc'] not in {0, 1, 2} or runner_summary['phase'] != 'terminal':
                raise RuntimeError('runner summary bounds differ')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            run_id = candidate_run_id
    runner_rc = process.wait()
finally:
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    root_key = hf_token = ''
    del root_key, hf_token, runner_env


In [ ]:
terminal_run_id = run_id or 'missing-run-id'
drive_run_dir = artifact_sink / terminal_run_id
zip_path = drive_run_dir / f'{terminal_run_id}.zip'
checksum_path = drive_run_dir / f'{terminal_run_id}.zip.sha256'
pair_present = zip_path.is_file() and checksum_path.is_file()
summary = {'run_id': run_id, 'resolved_exact': resolved_exact, 'runner_rc': runner_rc, 'zip_path': str(zip_path), 'checksum_path': str(checksum_path), 'pair_present': pair_present}
print(summary)
if runner_summary is None:
    raise RuntimeError('runner did not provide a bounded terminal summary')
if run_id is None:
    raise RuntimeError('runner produced no deterministic run identity')
if runner_summary['run_id'] != run_id or runner_summary['rc'] != runner_rc:
    raise RuntimeError('runner terminal identity or return code differs')
if not pair_present:
    raise RuntimeError('runner did not leave the expected terminal package pair')
if runner_rc != 0:
    raise RuntimeError('runner completed with retained operational failures')
